In [266]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import sklearn.linear_model as lm
import statsmodels.formula.api as smf
import statsmodels.stats.multicomp as mc
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.model_selection import train_test_split
from statsmodels.tools.tools import add_constant
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import LassoCV
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

#### After collecting the scores of students, we'll be analyzing the effect on various targets & factors.

### Section 1: MANOVA

In [195]:
df = pd.read_csv('/Users/Unicorn/Downloads/responses.csv')
df.head(2)

,Timestamp,are you motivated to study?,where do you usually like to study? (choose one that works best),what time of the day works best for you for studying?,"how are you usually doing, health-wise?",do you have a formal exercise routine?,how much is your screen time? (enter number of hours),how positive is your environment around you?,which study method works best for you?,"do you have any extracurricular activities? (like internship, work, sports tournaments? does not include gym)","how many hours do you usually study for exams? (only enter number of hours, per day wise)",do you like to study w/ music?,"lastly, please enter your exam score (out of 100)\n(preferably in a subject like physics)"
0,3/29/2024 16:12:33,neutral,cafe,night,fit,No,10.0,super positive,"writing down, listening",No,8,Yes,85
1,3/29/2024 16:32:52,neutral,cafe,past midnight,alright,No,10.0,positive,writing down,No,8,Yes,85


In [196]:
df = df.drop('Timestamp',axis=1)

In [197]:
df.columns = ['motiv','place','time',
              'health','exercise','screentime','env','method',
              'extra', 'hours','music','score']

In [198]:
df.head()

,motiv,place,time,health,exercise,screentime,env,method,extra,hours,music,score
0,neutral,cafe,night,fit,No,10.0,super positive,"writing down, listening",No,8,Yes,85
1,neutral,cafe,past midnight,alright,No,10.0,positive,writing down,No,8,Yes,85
2,no,at home,early morning,alright,No,12.0,super positive,writing down,Yes,4,Maybe,86
3,no,at home,morning,alright,Yes,6.0,neutral,writing down,Yes,4,Yes,60
4,no,at home,past midnight,alright,Yes,6.0,neutral,passive reading,No,10,No,60


In [199]:
df['screentime'] = df['screentime'].astype('int64')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   motiv       78 non-null     object
 1   place       78 non-null     object
 2   time        78 non-null     object
 3   health      78 non-null     object
 4   exercise    78 non-null     object
 5   screentime  78 non-null     int64 
 6   env         78 non-null     object
 7   method      78 non-null     object
 8   extra       78 non-null     object
 9   hours       78 non-null     int64 
 10  music       78 non-null     object
 11  score       78 non-null     int64 
dtypes: int64(3), object(9)
memory usage: 7.4+ KB


## Score, Screentime, Hours: MANOVA

In [200]:
#conducting manova on all the continuous variables
maov = MANOVA.from_formula('screentime+score+hours~motiv+place+time+health+exercise+env+method+extra+music',data=df)
print(maov.mv_test())

                  Multivariate linear model
                                                              
--------------------------------------------------------------
         Intercept        Value  Num DF  Den DF F Value Pr > F
--------------------------------------------------------------
            Wilks' lambda 0.4569 3.0000 53.0000 20.9980 0.0000
           Pillai's trace 0.5431 3.0000 53.0000 20.9980 0.0000
   Hotelling-Lawley trace 1.1886 3.0000 53.0000 20.9980 0.0000
      Roy's greatest root 1.1886 3.0000 53.0000 20.9980 0.0000
--------------------------------------------------------------
                                                              
--------------------------------------------------------------
          motiv          Value  Num DF  Den DF  F Value Pr > F
--------------------------------------------------------------
           Wilks' lambda 0.7753 6.0000 106.0000  2.3976 0.0327
          Pillai's trace 0.2359 6.0000 108.0000  2.4068 0.0319
  Hotelling

 #### motivation impacts all the 3 variables.

In [201]:
#conducting individual anova
fit1 = ols('score~motiv',data=df).fit()
anova1 = sm.stats.anova_lm(fit1,typ=1)

fit2 = ols('screentime~motiv',data=df).fit()
anova2 = sm.stats.anova_lm(fit2,typ=1)

fit3 = ols('hours~motiv',data=df).fit()
anova3 = sm.stats.anova_lm(fit3,typ=1)

In [202]:
print('score: ')
print(anova1)
print()
print('screentime: ')
print(anova2)
print()
print('hours: ')
print(anova3)

score: 
            df        sum_sq     mean_sq         F    PR(>F)
motiv      2.0    358.257046  179.128523  1.054321  0.353535
Residual  75.0  12742.460902  169.899479       NaN       NaN

screentime: 
            df      sum_sq    mean_sq         F    PR(>F)
motiv      2.0   23.576619  11.788310  0.994184  0.374848
Residual  75.0  889.295175  11.857269       NaN       NaN

hours: 
            df      sum_sq    mean_sq         F   PR(>F)
motiv      2.0   85.326475  42.663237  4.789158  0.01103
Residual  75.0  668.122243   8.908297       NaN      NaN


In [203]:
#motivation only impacts the number of hours a student studies.

In [204]:
tukey1 = mc.MultiComparison(df['hours'],df['motiv'])
tukeyans = tukey1.tukeyhsd().summary()
tukeyans

group1,group2,meandiff,p-adj,lower,upper,reject
neutral,no,-0.1173,0.9896,-2.151,1.9164,False
neutral,yes,2.2226,0.0172,0.3312,4.114,True
no,yes,2.3399,0.0336,0.1484,4.5315,True


#### i.e there is a significant difference in the number of study hours between students that are neutral and have motivation, and also between students that have no motivation at all and are motivated.

In [205]:
df_motiv_yes = df.loc[df['motiv']=='yes']
df_motiv_neu = df.loc[df['motiv']=='neutral']
df_motiv_no = df.loc[df['motiv']=='no']

In [206]:
df_motivyes_hours = df_motiv_yes['hours']
df_motivno_hours = df_motiv_no['hours']
df_motivneu_hours = df_motiv_neu['hours']

In [263]:
#t-test
from scipy import stats
from scipy.stats import t
print('between neutral and motivated: ')
print()
#H0: study hours neu <= motivated
#H1: study hours neu > motivated
print(stats.ttest_ind(df_motivneu_hours,df_motivyes_hours,alternative='greater'))
print()
print('between not motivated and motivated: ')
print()
#H0: study hours no <= motivated
#H1: study hours no > motivated
print(stats.ttest_ind(df_motivno_hours,df_motivyes_hours,alternative='greater'))
print()

between neutral and motivated: 

Ttest_indResult(statistic=-2.777405179714199, pvalue=0.996298047418201)

between not motivated and motivated: 

Ttest_indResult(statistic=-2.435354380780943, pvalue=0.9903434596669561)



In [267]:
#p-value < 0.05: reject null
#since. p-val is > 0.05, we accept null hypothesis
#i.e in both the cases, study hours for motivated students is more.

### Students with motivation study for more number of hours.

## Score & Screentime: MANOVA

In [208]:
maov2 = MANOVA.from_formula('screentime+score~motiv+place+time+health+exercise+env+method+extra+music',data=df)
print(maov2.mv_test())

                  Multivariate linear model
                                                             
-------------------------------------------------------------
        Intercept        Value  Num DF  Den DF F Value Pr > F
-------------------------------------------------------------
           Wilks' lambda 0.4634 2.0000 54.0000 31.2597 0.0000
          Pillai's trace 0.5366 2.0000 54.0000 31.2597 0.0000
  Hotelling-Lawley trace 1.1578 2.0000 54.0000 31.2597 0.0000
     Roy's greatest root 1.1578 2.0000 54.0000 31.2597 0.0000
-------------------------------------------------------------
                                                             
-------------------------------------------------------------
         motiv          Value  Num DF  Den DF  F Value Pr > F
-------------------------------------------------------------
          Wilks' lambda 0.8495 4.0000 108.0000  2.2945 0.0639
         Pillai's trace 0.1561 4.0000 110.0000  2.3276 0.0607
 Hotelling-Lawley trace 0.

#### environment affects both score and screentime.

In [209]:
fit4 = ols('screentime~env',data=df).fit()
anova4 = sm.stats.anova_lm(fit4,typ=1)
fit5 = ols('score~env',data=df).fit()
anova5 = sm.stats.anova_lm(fit5,typ=1)
print('screentime: ')
print(anova4)
print()
print('score: ')
print(anova5)
print()

screentime: 
            df      sum_sq    mean_sq         F    PR(>F)
env        4.0  100.771795  25.192949  2.264604  0.070354
Residual  73.0  812.100000  11.124658       NaN       NaN

score: 
            df        sum_sq     mean_sq         F    PR(>F)
env        4.0    927.277770  231.819443  1.390143  0.245823
Residual  73.0  12173.440179  166.759455       NaN       NaN



In [265]:
#none affected individually.

### Section 2: ANOVA for Score 

#### as we know from manova and individual anovas, env and motivation have no significant impact on the score, hence we can disregard the two columns.

In [213]:
df.columns

Index(['motiv', 'place', 'time', 'health', 'exercise', 'screentime', 'env',
       'method', 'extra', 'hours', 'music', 'score'],
      dtype='object')

In [214]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   motiv       78 non-null     object
 1   place       78 non-null     object
 2   time        78 non-null     object
 3   health      78 non-null     object
 4   exercise    78 non-null     object
 5   screentime  78 non-null     int64 
 6   env         78 non-null     object
 7   method      78 non-null     object
 8   extra       78 non-null     object
 9   hours       78 non-null     int64 
 10  music       78 non-null     object
 11  score       78 non-null     int64 
dtypes: int64(3), object(9)
memory usage: 7.4+ KB


In [215]:
anova_fit = ols('score~place+time+health+exercise+screentime+method+extra+hours+music',data=df).fit()
anova_6 = sm.stats.anova_lm(anova_fit,typ=1)
print(anova_6)

              df       sum_sq     mean_sq         F    PR(>F)
place        2.0   386.697899  193.348949  1.165752  0.318763
time         4.0   513.937621  128.484405  0.774667  0.546084
health       2.0  1302.547011  651.273505  3.926702  0.025061
exercise     1.0    76.619522   76.619522  0.461960  0.499368
method       4.0   562.742373  140.685593  0.848231  0.500400
extra        1.0    34.833096   34.833096  0.210018  0.648437
music        2.0    84.164563   42.082281  0.253725  0.776747
screentime   1.0   148.580846  148.580846  0.895834  0.347762
hours        1.0   204.994063  204.994063  1.235964  0.270760
Residual    59.0  9785.600955  165.857643       NaN       NaN


#### only health impacts the score.

In [216]:
tukey3 = mc.MultiComparison(df['score'],df['health'])
tukeyans3 = tukey3.tukeyhsd().summary()
tukeyans3

group1,group2,meandiff,p-adj,lower,upper,reject
alright,fit,-7.6444,0.0382,-14.9501,-0.3388,True
alright,mostly ill,-1.6444,0.9592,-15.9523,12.6634,False
fit,mostly ill,6.0,0.5957,-8.7358,20.7358,False


#### i.e there is a significant difference in the scores of students who are alright and fit.

In [217]:
df_alr = df.loc[df['health']=='alright']
df_fit = df.loc[df['health']=='fit']

In [218]:
df_alr_sco = df_alr['score']
df_fit_sco = df_fit['score']

In [264]:
#t-test
print('between students who are alright and fit, health wise: ')
print()
#H0: score of alright <= fit
#H1: score of alright > fit
print(stats.ttest_ind(df_alr_sco,df_fit_sco,alternative='greater'))

between students who are alright and fit, health wise: 

Ttest_indResult(statistic=2.524567970290458, pvalue=0.006910347275658516)


In [268]:
#p-value is less than 0.05, we reject null.

#### students that are alright have better scores.

In [220]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   motiv       78 non-null     object
 1   place       78 non-null     object
 2   time        78 non-null     object
 3   health      78 non-null     object
 4   exercise    78 non-null     object
 5   screentime  78 non-null     int64 
 6   env         78 non-null     object
 7   method      78 non-null     object
 8   extra       78 non-null     object
 9   hours       78 non-null     int64 
 10  music       78 non-null     object
 11  score       78 non-null     int64 
dtypes: int64(3), object(9)
memory usage: 7.4+ KB


In [221]:
df.isna().sum()

motiv         0
place         0
time          0
health        0
exercise      0
screentime    0
env           0
method        0
extra         0
hours         0
music         0
score         0
dtype: int64

In [222]:
df['health'].value_counts()

alright       45
fit           28
mostly ill     5
Name: health, dtype: int64

### Section 3: Regression Model

In [223]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   motiv       78 non-null     object
 1   place       78 non-null     object
 2   time        78 non-null     object
 3   health      78 non-null     object
 4   exercise    78 non-null     object
 5   screentime  78 non-null     int64 
 6   env         78 non-null     object
 7   method      78 non-null     object
 8   extra       78 non-null     object
 9   hours       78 non-null     int64 
 10  music       78 non-null     object
 11  score       78 non-null     int64 
dtypes: int64(3), object(9)
memory usage: 7.4+ KB


In [224]:
fitori = smf.ols('score~motiv+place+time+health+exercise+screentime+env+method+extra+hours+music',data=df).fit()
fitori.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  score   R-squared:                       0.395
Model:                            OLS   Adj. R-squared:                  0.121
Method:                 Least Squares   F-statistic:                     1.443
Date:                Wed, 03 Apr 2024   Prob (F-statistic):              0.133
Time:                        09:46:37   Log-Likelihood:                -290.89
No. Observations:                  78   AIC:                             631.8
Df Residuals:                      53   BIC:                             690.7
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
=====================================================================================================
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            93.6197     14.504      6.455      0.000      64.529     122.711
motiv[T.no]                          -5.8248      4.607     -1.264      0.212     -15.065       3.415
motiv[T.yes]                          4.7306      3.749      1.262      0.213      -2.789      12.251
place[T.cafe]                         7.2978      6.529      1.118      0.269      -5.797      20.393
place[T.library]                      0.8963      4.700      0.191      0.849      -8.531      10.324
time[T.evening]                      11.4031      6.239      1.828      0.073      -1.110      23.916
time[T.morning]                       4.3616      5.942      0.734      0.466      -7.557      16.280
time[T.night]                         9.9454      5.896      1.687      0.098      -1.880      21.771
time[T.past midnight]                11.1203      5.974      1.861      0.068      -0.863      23.103
health[T.fit]                       -10.4755      3.629     -2.887      0.006     -17.754      -3.197
health[T.mostly ill]                -10.3748      8.888     -1.167      0.248     -28.202       7.452
exercise[T.Yes]                      -2.3689      3.386     -0.700      0.487      -9.161       4.423
env[T.neutral]                      -13.2043     10.422     -1.267      0.211     -34.107       7.699
env[T.positive]                      -6.4684      9.855     -0.656      0.514     -26.236      13.299
env[T.really negative]              -37.3369     18.294     -2.041      0.046     -74.030      -0.644
env[T.super positive]                 0.8456     11.431      0.074      0.941     -22.083      23.774
method[T.passive reading]            -6.7592      9.332     -0.724      0.472     -25.477      11.958
method[T.visualising]                -5.8041      9.059     -0.641      0.524     -23.973      12.365
method[T.writing down]               -9.2254      8.694     -1.061      0.293     -26.663       8.213
method[T.writing down, listening]   -13.9636     18.248     -0.765      0.448     -50.565      22.638
extra[T.Yes]                         -0.5420      3.497     -0.155      0.877      -7.556       6.472
music[T.No]                          -2.5144      4.691     -0.536      0.594     -11.923       6.894
music[T.Yes]                         -5.6527      4.055     -1.394      0.169     -13.785       2.480
screentime                            0.1602      0.503      0.318      0.752      -0.849       1.170
hours                                 0.2227      0.548      0.406      0.686      -0.876       1.322
==============================================================================
Omnibus:                       25.860   Durbin-Watson:                   1.866
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            

In [225]:
df1 = df

In [226]:
df1['method'] = df1['method'].str.replace('writing down, listening','listening')

In [227]:
df['method'] = df['method'].str.replace('writing down, listening','listening')

In [228]:
df['method'].value_counts()

writing down       42
visualising        18
passive reading    14
listening           4
Name: method, dtype: int64

#### Step 1: converting int -> float, and getting dummies

In [229]:
df1 = pd.get_dummies(df1, columns= ['motiv','place','time','health','exercise','env','method','extra','music']).astype(float)

In [230]:
df1

,screentime,hours,score,motiv_neutral,motiv_no,motiv_yes,place_at home,place_cafe,place_library,time_early morning,...,env_super positive,method_listening,method_passive reading,method_visualising,method_writing down,extra_No,extra_Yes,music_Maybe,music_No,music_Yes
0,10.0,8.0,85.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,10.0,8.0,85.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
2,12.0,4.0,86.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
3,6.0,4.0,60.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
4,6.0,10.0,60.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,7.0,5.0,97.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
74,6.0,5.0,81.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
75,5.0,2.0,88.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
76,8.0,10.0,70.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0


In [231]:
y=df1.score
X=df1.drop(['score'],axis=1)
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   screentime              78 non-null     float64
 1   hours                   78 non-null     float64
 2   motiv_neutral           78 non-null     float64
 3   motiv_no                78 non-null     float64
 4   motiv_yes               78 non-null     float64
 5   place_at home           78 non-null     float64
 6   place_cafe              78 non-null     float64
 7   place_library           78 non-null     float64
 8   time_early morning      78 non-null     float64
 9   time_evening            78 non-null     float64
 10  time_morning            78 non-null     float64
 11  time_night              78 non-null     float64
 12  time_past midnight      78 non-null     float64
 13  health_alright          78 non-null     float64
 14  health_fit              78 non-null     floa

#### Step 3: Using alphas, ElasticNet to drop columns

In [232]:
alphas = 10**np.linspace(10,-2,100)*0.5

In [233]:
xtrain,xtest,ytrain,ytest = train_test_split(X,y,test_size=0.20,random_state=0)

In [234]:
fitCVEN = ElasticNetCV(alphas=alphas)
fitCVEN.fit(X,y)
best_a = fitCVEN.alpha_
fitEN = ElasticNet(alpha=best_a)
fitEN.fit(xtrain,ytrain)
fitEN.coef_

array([ 0.02315475,  0.45890376,  0.        , -0.00896619,  0.        ,
       -0.        ,  0.        , -0.        , -0.        ,  0.        ,
        0.        ,  0.        , -0.        ,  0.33548702, -0.5089929 ,
        0.        ,  0.10394587, -0.10394747,  0.        , -0.        ,
        0.        , -0.        ,  0.        ,  0.        , -0.        ,
        0.        , -0.        ,  0.        , -0.        , -0.        ,
        0.        , -0.        ])

In [235]:
fitCVL = LassoCV(alphas=alphas)
fitCVL.fit(xtrain,ytrain)
bestal = fitCVL.alpha_
fitL = Lasso(alpha=bestal)
fitL.fit(xtrain,ytrain)
pred = fitL.predict(xtrain)

In [236]:
from sklearn.metrics import mean_squared_error
fitEN = ElasticNet(alpha=best_a)
fitEN.fit(xtrain,ytrain)
predEN_train = fitEN.predict(xtrain)
print(mean_squared_error(ytrain,pred))

164.9513527575442


In [237]:
a = pd.Series(fitEN.coef_,index=X.columns)

In [238]:
fitencoef = pd.DataFrame(a,columns=['value'])
fitencoef

,value
screentime,0.023155
hours,0.458904
motiv_neutral,0.000000
motiv_no,-0.008966
motiv_yes,0.000000
place_at home,-0.000000
place_cafe,0.000000
place_library,-0.000000
time_early morning,-0.000000
time_evening,0.000000


In [239]:
import statsmodels.api as sm
import pandas as pd
fitscreen = smf.ols('screentime ~ hours', df).fit()
VIF_screen = 1 / (1 - fitscreen.rsquared)
print("VIF for 'screentime':", VIF_screen)
fithours = smf.ols('hours ~ screentime', df).fit()
VIF_hours = 1 / (1 - fithours.rsquared)
print("VIF for 'hours':", VIF_hours)

VIF for 'screentime': 1.000922636827748
VIF for 'hours': 1.0009226368277482


In [240]:
#independent continuous variables are not highly correlated.

In [241]:
#we can drop:
# motiv_neutral, motiv_yes, place_cafe, time_morning, time_eve,
# time_night, health_mostly_ill, env_neg, env_pos, env_superpos,
# method_listening, method_visualising, extra_no, music_no

In [242]:
#14 columns have been dropped.
#now we conduct our ols normally.

In [243]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   screentime              78 non-null     float64
 1   hours                   78 non-null     float64
 2   score                   78 non-null     float64
 3   motiv_neutral           78 non-null     float64
 4   motiv_no                78 non-null     float64
 5   motiv_yes               78 non-null     float64
 6   place_at home           78 non-null     float64
 7   place_cafe              78 non-null     float64
 8   place_library           78 non-null     float64
 9   time_early morning      78 non-null     float64
 10  time_evening            78 non-null     float64
 11  time_morning            78 non-null     float64
 12  time_night              78 non-null     float64
 13  time_past midnight      78 non-null     float64
 14  health_alright          78 non-null     floa

In [244]:
import pandas as pd
df1.rename(columns={'place_at home':'place_at_home'},inplace=True)
df1.rename(columns={'time_early morning':'time_early_morning'},inplace=True)
df1.rename(columns={'time_past midnight':'time_past_midnight'},inplace=True)
df1.rename(columns={'time_early morning':'time_early_morning'},inplace=True)
df1.rename(columns={'env_really negative':'env_really_negative'},inplace=True)
df1.rename(columns={'time_early morning':'time_early_morning'},inplace=True)
df1.rename(columns={'method_passive reading':'method_passive_reading'},inplace=True)
df1.rename(columns={'method_writing down':'method_writing_down'},inplace=True)
df1.rename(columns={'env_super positive':'env_super_positive'},inplace=True)
df1.rename(columns={'health_mostly ill':'health_mostly_ill'},inplace=True)

In [245]:
str(list(df1.columns)).strip("[]").replace("', '","+")

"'screentime+hours+score+motiv_neutral+motiv_no+motiv_yes+place_at_home+place_cafe+place_library+time_early_morning+time_evening+time_morning+time_night+time_past_midnight+health_alright+health_fit+health_mostly_ill+exercise_No+exercise_Yes+env_negative+env_neutral+env_positive+env_really_negative+env_super_positive+method_listening+method_passive_reading+method_visualising+method_writing_down+extra_No+extra_Yes+music_Maybe+music_No+music_Yes'"

In [246]:
fito = smf.ols('score~screentime+hours+motiv_neutral+motiv_no+motiv_yes+place_at_home+place_cafe+place_library+time_early_morning+time_evening+time_morning+time_night+time_past_midnight+health_alright+health_fit+health_mostly_ill+exercise_No+exercise_Yes+env_negative+env_neutral+env_positive+env_really_negative+env_super_positive+method_listening+method_passive_reading+method_visualising+method_writing_down+extra_No+extra_Yes+music_Maybe+music_No+music_Yes',data=df1).fit()
fito.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  score   R-squared:                       0.389
Model:                            OLS   Adj. R-squared:                  0.128
Method:                 Least Squares   F-statistic:                     1.492
Date:                Wed, 03 Apr 2024   Prob (F-statistic):              0.115
Time:                        09:46:37   Log-Likelihood:                -291.32
No. Observations:                  78   AIC:                             630.6
Df Residuals:                      54   BIC:                             687.2
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
==========================================================================================
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 18.7260      1.503     12.462      0.000      15.713      21.739
screentime                 0.1311      0.500      0.262      0.794      -0.871       1.133
hours                      0.2024      0.545      0.371      0.712      -0.891       1.296
motiv_neutral              6.4509      2.242      2.877      0.006       1.955      10.947
motiv_no                   0.6334      2.961      0.214      0.831      -5.303       6.570
motiv_yes                 11.6416      2.443      4.764      0.000       6.743      16.540
place_at_home              3.6766      2.918      1.260      0.213      -2.173       9.526
place_cafe                 9.8130      4.135      2.373      0.021       1.523      18.103
place_library              5.2364      2.958      1.770      0.082      -0.695      11.167
time_early_morning        -3.6278      4.234     -0.857      0.395     -12.116       4.860
time_evening               7.7111      3.523      2.189      0.033       0.649      14.774
time_morning               0.8810      2.952      0.298      0.766      -5.037       6.799
time_night                 5.9568      2.850      2.090      0.041       0.243      11.670
time_past_midnight         7.8048      3.240      2.409      0.019       1.309      14.301
health_alright            13.4737      3.181      4.235      0.000       7.095      19.852
health_fit                 2.7209      3.512      0.775      0.442      -4.320       9.762
health_mostly_ill          2.5313      6.098      0.415      0.680      -9.694      14.757
exercise_No               10.5620      1.739      6.072      0.000       7.075      14.049
exercise_Yes               8.1639      1.947      4.192      0.000       4.260      12.068
env_negative              16.8068      8.314      2.021      0.048       0.138      33.475
env_neutral                1.7639      3.996      0.441      0.661      -6.247       9.775
env_positive               8.8822      3.767      2.358      0.022       1.329      16.435
env_really_negative      -23.0044     12.005     -1.916      0.061     -47.074       1.065
env_super_positive        14.2774      5.705      2.503      0.015       2.839      25.715
method_listening           7.4037      5.455      1.357      0.180      -3.533      18.340
method_passive_reading     4.3509      3.544      1.228      0.225      -2.755      11.456
method_visualising         5.2161      3.155      1.653      0.104      -1.110      11.542
method_writing_down        1.7553      2.663      0.659      0.513      -3.583       7.094
extra_No                   9.2133      1.799      5.120      0.000       5.606      12.821
extra_Yes                  9.5126      1.833      5.190      0.000       5.838      13.187
music_Maybe                8.9837      2.560      3.509      0.001       3.851  

In [247]:
#dropping 14 columns
fito1 = smf.ols('score~screentime+hours+motiv_no+place_at_home+place_library+time_early_morning+time_past_midnight+health_alright+health_fit+exercise_No+exercise_Yes+env_neutral+env_really_negative+method_passive_reading+method_writing_down+extra_Yes+music_Maybe+music_Yes',data=df1).fit()
fito1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  score   R-squared:                       0.314
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     1.619
Date:                Wed, 03 Apr 2024   Prob (F-statistic):             0.0876
Time:                        09:46:37   Log-Likelihood:                -295.78
No. Observations:                  78   AIC:                             627.6
Df Residuals:                      60   BIC:                             670.0
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
==========================================================================================
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 58.5475      6.481      9.033      0.000      45.583      71.512
screentime                 0.1438      0.456      0.315      0.754      -0.768       1.056
hours                      0.3890      0.523      0.744      0.460      -0.657       1.435
motiv_no                  -6.1085      3.811     -1.603      0.114     -13.732       1.515
place_at_home             -6.8342      5.788     -1.181      0.242     -18.412       4.743
place_library             -4.8750      6.123     -0.796      0.429     -17.123       7.373
time_early_morning        -7.8175      5.203     -1.503      0.138     -18.224       2.589
time_past_midnight         2.0938      3.923      0.534      0.596      -5.754       9.941
health_alright             6.5985      6.575      1.004      0.320      -6.554      19.751
health_fit                -3.1724      6.808     -0.466      0.643     -16.791      10.446
exercise_No               30.7571      3.370      9.126      0.000      24.015      37.499
exercise_Yes              27.7905      3.888      7.147      0.000      20.013      35.568
env_neutral               -6.8427      3.199     -2.139      0.037     -13.242      -0.443
env_really_negative      -32.2124     14.396     -2.238      0.029     -61.008      -3.417
method_passive_reading    -2.2647      4.679     -0.484      0.630     -11.624       7.094
method_writing_down       -3.6345      3.553     -1.023      0.310     -10.741       3.472
extra_Yes                  0.8283      3.250      0.255      0.800      -5.672       7.329
music_Maybe                1.5156      4.618      0.328      0.744      -7.722      10.754
music_Yes                 -3.9665      4.299     -0.923      0.360     -12.565       4.632
==============================================================================
Omnibus:                       17.396   Durbin-Watson:                   1.966
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               21.737
Skew:                          -1.033   Prob(JB):                     1.90e-05
Kurtosis:                       4.556   Cond. No.                     7.97e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 1.19e-30. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [248]:
#dropping highest pval - extra_Yes
fitre = smf.ols('score~screentime+hours+motiv_no+place_at_home+place_library+time_early_morning+time_past_midnight+health_alright+health_fit+exercise_No+exercise_Yes+env_neutral+env_really_negative+method_passive_reading+method_writing_down+music_Maybe+music_Yes',data=df1).fit()
fitre.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  score   R-squared:                       0.314
Model:                            OLS   Adj. R-squared:                  0.134
Method:                 Least Squares   F-statistic:                     1.743
Date:                Wed, 03 Apr 2024   Prob (F-statistic):             0.0622
Time:                        09:46:37   Log-Likelihood:                -295.82
No. Observations:                  78   AIC:                             625.6
Df Residuals:                      61   BIC:                             665.7
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
==========================================================================================
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 58.6471      6.420      9.135      0.000      45.810      71.484
screentime                 0.1706      0.440      0.387      0.700      -0.710       1.051
hours                      0.3522      0.499      0.706      0.483      -0.645       1.349
motiv_no                  -5.9894      3.753     -1.596      0.116     -13.495       1.516
place_at_home             -6.6675      5.707     -1.168      0.247     -18.078       4.743
place_library             -4.7777      6.064     -0.788      0.434     -16.904       7.348
time_early_morning        -7.8932      5.154     -1.531      0.131     -18.200       2.413
time_past_midnight         1.9658      3.861      0.509      0.612      -5.755       9.686
health_alright             6.7384      6.502      1.036      0.304      -6.263      19.740
health_fit                -2.8627      6.648     -0.431      0.668     -16.155      10.430
exercise_No               30.7974      3.341      9.219      0.000      24.117      37.478
exercise_Yes              27.8497      3.851      7.231      0.000      20.148      35.551
env_neutral               -6.7787      3.165     -2.142      0.036     -13.107      -0.450
env_really_negative      -32.3750     14.271     -2.269      0.027     -60.912      -3.838
method_passive_reading    -2.4151      4.606     -0.524      0.602     -11.625       6.795
method_writing_down       -3.7674      3.487     -1.080      0.284     -10.741       3.206
music_Maybe                1.5553      4.580      0.340      0.735      -7.603      10.714
music_Yes                 -3.9309      4.263     -0.922      0.360     -12.456       4.594
==============================================================================
Omnibus:                       16.337   Durbin-Watson:                   1.967
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               19.709
Skew:                          -0.999   Prob(JB):                     5.25e-05
Kurtosis:                       4.440   Cond. No.                     8.67e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is  1e-30. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [249]:
#drop music_maybe
#drop health_fit
#drop screentime
#drop time_past_midnight
#drop method_passive_reading
#drop hours
#drop method_writing_down
#drop place_library
#drop place_at_home
#drop music_Yes
#drop time_early_morning
fitre = smf.ols('score~motiv_no+health_alright+exercise_No+exercise_Yes+env_neutral+env_really_negative',data=df1).fit()
fitre.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  score   R-squared:                       0.224
Model:                            OLS   Adj. R-squared:                  0.170
Method:                 Least Squares   F-statistic:                     4.160
Date:                Wed, 03 Apr 2024   Prob (F-statistic):            0.00224
Time:                        09:46:37   Log-Likelihood:                -300.60
No. Observations:                  78   AIC:                             613.2
Df Residuals:                      72   BIC:                             627.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              52.9139      1.549     34.156      0.000      49.826      56.002
motiv_no               -7.1621      3.244     -2.208      0.030     -13.628      -0.696
health_alright          8.6128      2.997      2.873      0.005       2.637      14.588
exercise_No            28.8466      1.752     16.467      0.000      25.355      32.339
exercise_Yes           24.0673      1.492     16.135      0.000      21.094      27.041
env_neutral            -6.3245      2.855     -2.215      0.030     -12.016      -0.633
env_really_negative   -30.3732     12.213     -2.487      0.015     -54.720      -6.026
==============================================================================
Omnibus:                       14.173   Durbin-Watson:                   1.936
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               16.304
Skew:                          -0.900   Prob(JB):                     0.000288
Kurtosis:                       4.334   Cond. No.                     1.04e+16
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 1.57e-30. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [250]:
fitfinal = smf.ols('score~motiv_no+health_alright+exercise_No+exercise_Yes+env_neutral+env_really_negative',data=df1).fit()
fitfinal.summary().tables[0]

Dep. Variable:,score,R-squared:,0.224
Model:,OLS,Adj. R-squared:,0.170
Method:,Least Squares,F-statistic:,4.160
Date:,"Wed, 03 Apr 2024",Prob (F-statistic):,0.00224
Time:,09:46:37,Log-Likelihood:,-300.60
No. Observations:,78,AIC:,613.2
Df Residuals:,72,BIC:,627.3
Df Model:,5,,
Covariance Type:,nonrobust,,


In [251]:
X.rename(columns={'place_at home':'place_at_home'},inplace=True)
X.rename(columns={'time_early morning':'time_early_morning'},inplace=True)
X.rename(columns={'time_past midnight':'time_past_midnight'},inplace=True)
X.rename(columns={'time_early morning':'time_early_morning'},inplace=True)
X.rename(columns={'env_really negative':'env_really_negative'},inplace=True)
X.rename(columns={'time_early morning':'time_early_morning'},inplace=True)
X.rename(columns={'method_passive reading':'method_passive_reading'},inplace=True)
X.rename(columns={'method_writing down':'method_writing_down'},inplace=True)
X.rename(columns={'env_super positive':'env_super_positive'},inplace=True)
X.rename(columns={'health_mostly ill':'health_mostly_ill'},inplace=True)

In [252]:
xtrain,xtest,ytrain,ytest = train_test_split(X,y,test_size=0.20,random_state=123)

In [253]:
xtrain_new = xtrain[['motiv_no','health_alright','exercise_No','exercise_Yes','env_neutral','env_really_negative']]
ytrain_new = ytrain
xtest_new = xtest[['motiv_no','health_alright','exercise_No','exercise_Yes','env_neutral','env_really_negative']]
ytest_new = ytest

In [254]:
p1 = fitfinal.predict(xtrain_new)
mse1 = np.mean((ytrain_new-p1)**2)
print(mse1)
p2 = fitfinal.predict(xtest_new)
mse2 = np.mean((ytest_new-p2)**2)
print(mse2)

121.99663448499827
162.53051504828335


In [255]:
p1

71    81.760448
43    79.269498
51    90.373221
1     90.373221
37    81.760448
        ...    
73    83.211090
47    85.594002
57    84.048717
17    90.373221
66    84.048717
Length: 62, dtype: float64

In [256]:
p2

53    79.269498
64    85.594002
70    79.269498
4     72.107367
60    72.107367
23    90.373221
29    84.048717
61    76.981229
8     70.656725
75    83.211090
9     74.598317
76    75.435944
44    76.981229
24    90.373221
77    70.656725
63    81.760448
dtype: float64